# Part D: Boosting and Leakage-Safe Pipelines
This notebook benchmarks XGBoost, LightGBM, and the Random Forest baseline on the breast-cancer data set solely as a technical classification benchmark. It is not a clinical or diagnostic system.

In [1]:
import json
import sys
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

project_root = Path.cwd().resolve()
if not (project_root / 'src').is_dir():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.evaluation import ModelEvaluator

np.random.seed(42)

## Data and Pipeline Setup
The data is split once with the same fixed seed and stratification used in notebook 02. No preprocessing is applied before the split.

In [2]:
dataset = load_breast_cancer()
X = dataset.data
y = dataset.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

All imputation and scaling live inside each Pipeline, so every cross-validation fold fits preprocessing only on its own training partition. This prevents preprocessing leakage. Scaling is not strictly necessary for tree-based boosters, but it is intentionally included to keep the pipeline interface consistent.

In [3]:
xgb_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),
    ('classifier', XGBClassifier(eval_metric='logloss', random_state=42, n_jobs=1)),
])
lgbm_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),
    ('classifier', LGBMClassifier(random_state=42, n_jobs=1, verbosity=-1)),
])
baseline_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=1)),
])

## Controlled Hyperparameter Search
Each search uses eight sampled combinations, five-fold cross-validation, and F1 scoring on the training set only.

In [4]:
xgb_search = RandomizedSearchCV(
    xgb_pipeline,
    param_distributions={
        'classifier__n_estimators': [100, 200],
        'classifier__max_depth': [3, 5],
        'classifier__learning_rate': [0.03, 0.1],
        'classifier__subsample': [0.8, 1.0],
    },
    n_iter=8, cv=5, scoring='f1', random_state=42, n_jobs=1
)
lgbm_search = RandomizedSearchCV(
    lgbm_pipeline,
    param_distributions={
        'classifier__n_estimators': [100, 200],
        'classifier__max_depth': [3, 5],
        'classifier__learning_rate': [0.03, 0.1],
        'classifier__subsample': [0.8, 1.0],
        'classifier__num_leaves': [15, 31],
    },
    n_iter=8, cv=5, scoring='f1', random_state=42, n_jobs=1
)

This output records the best hyperparameters found for each tuned boosting pipeline.

In [5]:
search_start = time.perf_counter()
xgb_search.fit(X_train, y_train)
xgb_search_time = time.perf_counter() - search_start
search_start = time.perf_counter()
lgbm_search.fit(X_train, y_train)
lgbm_search_time = time.perf_counter() - search_start
print('XGBoost best parameters:', xgb_search.best_params_)
print('LightGBM best parameters:', lgbm_search.best_params_)

XGBoost best parameters: {'classifier__subsample': 0.8, 'classifier__n_estimators': 200, 'classifier__max_depth': 5, 'classifier__learning_rate': 0.1}
LightGBM best parameters: {'classifier__subsample': 1.0, 'classifier__num_leaves': 31, 'classifier__n_estimators': 200, 'classifier__max_depth': 5, 'classifier__learning_rate': 0.03}


The selected parameter sets maximize mean cross-validated F1 among the controlled candidates. The search time is separate from the final single-fit training time reported in the model comparison.

## Model Comparison
Random Forest was the business-cost selection in notebook 02, so it is recreated here as the classical comparison baseline. ModelEvaluator supplies every test metric and business cost.

In [6]:
fn_cost = 5
fp_cost = 1
evaluator = ModelEvaluator(fn_weight=fn_cost, fp_weight=fp_cost)
best_xgb_pipeline = xgb_search.best_estimator_
best_lgbm_pipeline = lgbm_search.best_estimator_
comparison_models = {
    'XGBoost (tuned)': (best_xgb_pipeline, xgb_search),
    'LightGBM (tuned)': (best_lgbm_pipeline, lgbm_search),
    'Random Forest baseline': (baseline_pipeline, None),
}
comparison_records = []
fitted_pipelines = {}
for name, (pipeline, search) in comparison_models.items():
    train_start = time.perf_counter()
    pipeline.fit(X_train, y_train)
    training_time = time.perf_counter() - train_start
    prediction_start = time.perf_counter()
    predicted_labels = pipeline.predict(X_test)
    predicted_probabilities = pipeline.predict_proba(X_test)[:, 1]
    prediction_time = time.perf_counter() - prediction_start
    test_metrics = evaluator.evaluate(y_test, predicted_labels, predicted_probabilities)
    train_predictions = pipeline.predict(X_train)
    train_metrics = evaluator.evaluate(y_train, train_predictions, pipeline.predict_proba(X_train)[:, 1])
    if search is None:
        cv_f1 = np.nan
        cv_f1_std = np.nan
    else:
        best_index = search.best_index_
        cv_f1 = search.cv_results_['mean_test_score'][best_index]
        cv_f1_std = search.cv_results_['std_test_score'][best_index]
    comparison_records.append({
        'model': name, 'cv_f1': f'{cv_f1:.3f} ± {cv_f1_std:.3f}' if search is not None else 'Not tuned',
        'cv_f1_mean': cv_f1, 'cv_f1_std': cv_f1_std,
        'train_f1': train_metrics['f1'], 'accuracy': test_metrics['accuracy'],
        'precision': test_metrics['precision'], 'recall': test_metrics['recall'],
        'f1': test_metrics['f1'], 'roc_auc': test_metrics['roc_auc'],
        'training_time_seconds': training_time, 'prediction_time_seconds': prediction_time,
        'business_cost': test_metrics['business_cost']
    })
    fitted_pipelines[name] = pipeline
comparison_table = pd.DataFrame(comparison_records).sort_values(['f1', 'business_cost'], ascending=[False, True]).reset_index(drop=True)
comparison_table

,model,cv_f1,cv_f1_mean,cv_f1_std,train_f1,accuracy,precision,recall,f1,roc_auc,training_time_seconds,prediction_time_seconds,business_cost
0,XGBoost (tuned),0.976 ± 0.012,0.975959,0.012167,1.0,0.972028,0.957447,1.000000,0.978261,0.996226,0.227681,0.003680,4
1,LightGBM (tuned),0.974 ± 0.016,0.973904,0.015991,1.0,0.972028,0.967391,0.988889,0.978022,0.991195,0.142738,0.003762,8
2,Random Forest baseline,Not tuned,NaN,NaN,1.0,0.958042,0.956522,0.977778,0.967033,0.994969,0.503233,0.023314,14


The comparison contains the tuned boosters and the notebook 02 Random Forest baseline. CV F1 is shown as a mean and standard deviation for the searches, while all holdout metrics and costs come from ModelEvaluator.

This output compares each model's training and test F1 to identify potential overfitting gaps.

In [7]:
overfitting_table = comparison_table[['model', 'train_f1', 'f1']].copy()
overfitting_table['f1_gap'] = overfitting_table['train_f1'] - overfitting_table['f1']
overfitting_table['observation'] = np.where(overfitting_table['f1_gap'] > 0.05, 'Large gap: investigate overfitting', 'Small gap: generalization appears stable')
overfitting_table

,model,train_f1,f1,f1_gap,observation
0,XGBoost (tuned),1.0,0.978261,0.021739,Small gap: generalization appears stable
1,LightGBM (tuned),1.0,0.978022,0.021978,Small gap: generalization appears stable
2,Random Forest baseline,1.0,0.967033,0.032967,Small gap: generalization appears stable


A large train-to-test F1 gap flags possible overfitting; a small gap indicates the fitted model generalizes similarly to the holdout set. The table reports this observation separately for all three models.

## Required Discussion

### Boosting versus bagging
Bagging trains models independently and averages them to reduce variance, as in Random Forest. Boosting trains learners sequentially so later learners focus on previous errors, often reducing bias but requiring more care to avoid overfitting.

### Random Forest versus XGBoost
Random Forest is a bagged collection of decorrelated decision trees, while XGBoost is a regularized gradient-boosting method that builds trees sequentially. XGBoost exposes richer optimization controls, whereas Random Forest is commonly a robust lower-tuning baseline.

### XGBoost versus LightGBM
Both are gradient-boosted tree systems, but LightGBM uses a histogram-based, leaf-wise growth strategy designed for efficient large-scale training. XGBoost commonly uses level-wise growth and extensive regularization options, so their speed and accuracy trade-offs can vary by data set.

### Learning rate versus number of estimators
A lower learning rate makes each boosting step smaller and commonly requires more estimators to reach similar fit. A higher learning rate can converge with fewer trees but may overshoot useful solutions or overfit more readily.

### Why pipelines reduce leakage risk
A pipeline fits imputation and scaling only within the training portion of each cross-validation fold. This prevents validation samples from influencing preprocessing statistics and makes cross-validation estimates more trustworthy.

### Scaling for SVM and tree-based models
As noted in notebook 02, SVM depends on distances and kernels, so unscaled features can dominate its geometry. Tree-based models split one feature at a time and are therefore not scale-sensitive in the same way, although scaling remains in these pipelines for a consistent interface.

## Model Selection and Artifacts
The final model is selected from the F1-then-business-cost ordering, not accuracy alone. This favors strong balanced classification performance while respecting the five-to-one false-negative cost.

This output states the selected model and its statistical and business-cost justification.

In [8]:
selected_record = comparison_table.iloc[0]
selected_model_name = selected_record['model']
highest_accuracy_record = comparison_table.loc[comparison_table['accuracy'].idxmax()]
selection_reason = (
    f"Selected for F1={selected_record['f1']:.3f} and business cost={selected_record['business_cost']:.1f}; "
    f"highest accuracy belongs to {highest_accuracy_record['model']}."
)
print(f"Selected model: {selected_model_name}")
print(selection_reason)

Selected model: XGBoost (tuned)
Selected for F1=0.978 and business cost=4.0; highest accuracy belongs to XGBoost (tuned).


The named model is selected because its combined F1 and business-cost position is preferable under the stated cost weights. The comparison explicitly checks the highest-accuracy model so accuracy does not silently become the sole selection criterion.

The selected object is a complete fitted Pipeline containing imputation, scaling, and the classifier. Joblib is used directly because this artifact contract requires loading the Pipeline itself; SklearnModelTrainer.save stores a wrapper dictionary containing configuration in addition to its estimator.

In [9]:
artifact_directory = Path('artifacts')
artifact_directory.mkdir(exist_ok=True)
selected_pipeline = fitted_pipelines[selected_model_name]
joblib.dump(selected_pipeline, artifact_directory / 'model.joblib')
metrics_payload = {
    'selected_model': str(selected_model_name),
    'selection_reason': selection_reason,
    'comparison': json.loads(comparison_table.to_json(orient='records')),
}
with (artifact_directory / 'metrics.json').open('w', encoding='utf-8') as metrics_file:
    json.dump(metrics_payload, metrics_file, indent=2)

The artifacts directory now contains the selected full Pipeline and a JSON-serializable comparison record for all three models. No deep-learning models are included in this notebook.